## **Tensorflow Keras का उपयोग करके Optimization (अनुकूलन)**
इस hackerrank scenario में, आप **Optimisation (अनुकूलन)** के बारे में और मौजूद अलग-अलग optimizers के बारे में और उन्हें कैसे उपयोग करना है, यह जानेंगे।

हम यहाँ तीन scenarios देखेंगे —
1. अलग-अलग optimisation algorithms को tune (समायोजित) करना
2. SGD optimizer की learning rate और momentum को tune करना
3. Adam optimizer के beta values को tune करना

**नोट (Note)** - चुनौती (challenge) पूरी होने के बाद अंत में kernel को restart करके सभी cells को दोबारा run करें


### आवश्यक (required) packages import करें


In [ ]:
import numpy as np                                                # numpy import की — संख्यात्मक (numerical) और array संबंधित कामों के लिए
from sklearn.model_selection import GridSearchCV                  # GridSearchCV import किया — अलग-अलग hyperparameters को व्यवस्थित (systematic) तरीके से आज़माने और सबसे अच्छा combination ढूँढने के लिए
from keras.models import Sequential                                # Sequential import किया — layer-by-layer neural network model बनाने के लिए
from keras.layers import Dense                                     # Dense import किया — fully-connected (पूरी तरह जुड़ी हुई) layer बनाने के लिए
from keras.wrappers.scikit_learn import KerasClassifier            # KerasClassifier import किया — Keras model को sklearn के tools (जैसे GridSearchCV) के साथ उपयोग करने लायक बनाने के लिए
from sklearn.datasets import load_iris                              # load_iris import किया — प्रसिद्ध (famous) Iris फूलों का dataset लोड करने के लिए
from sklearn.model_selection import train_test_split                # train_test_split import किया — data को train/test हिस्सों में बाँटने के लिए
from sklearn.preprocessing import LabelEncoder,StandardScaler       # LabelEncoder और StandardScaler import किए — labels को encode करने और features को standardize (scale) करने के लिए
from sklearn.utils import shuffle                                   # shuffle import किया — data की rows को randomly फेरबदल (मिलाने) के लिए
from keras.utils.np_utils import to_categorical                     # to_categorical import किया — class labels (0,1,2) को one-hot encoding में बदलने के लिए
from keras.optimizers import SGD,Adam                                # SGD और Adam optimizers import किए — network के weights update करने के अलग-अलग तरीके
from matplotlib import pyplot                                       # pyplot import किया — ग्राफ/चार्ट बनाने के लिए
import seaborn as sns                                                # seaborn import किया — सुंदर सांख्यिकीय (statistical) plots बनाने के लिए
import pandas as pd                                                  # pandas import की — data को table (DataFrame) के रूप में संभालने के लिए
from keras.models import model_from_json                             # model_from_json import किया — JSON फ़ाइल से सहेजे (saved) हुए model को वापस लोड करने के लिए


### Dataset लोड करें

- `load_iris()` function का उपयोग करके iris dataset लोड करें।
- Iris dataset का data variable **X** में store करें।
- Iris dataset का target variable **y** में store करें।
- variable **y** को `to_categorial` function का उपयोग करके categorial variable (श्रेणीबद्ध चर) में बदलें और उसे variable **y** में save करें।
- variable **seed** में seed value 7 सेट करें और numpy के `random.seed` function का उपयोग करके seed value सेट करें।
- अब **X** और **y** data को `shuffle` function का उपयोग करके शफल (फेरबदल) करें और उसे variables **X**, **Y** में save करें।


In [ ]:
# fix random seed for reproducibility

iris = load_iris()                          # sklearn से Iris dataset लोड किया — फूलों के 4 features (लंबाई/चौड़ाई) और उनकी 3 species (किस्में) वाला मशहूर dataset
X = iris.data                                # dataset के input features (4 columns: sepal/petal की लंबाई-चौड़ाई) निकाले, X में रखे
#Y = to_categorical(iris.target,3)
y = iris.target                              # dataset के असली (actual) target labels (0, 1, 2 — तीन species) निकाले
y = to_categorical(y, 3)                     # y को one-hot encoding में बदला — यानी 0 -> [1,0,0], 1 -> [0,1,0], 2 -> [0,0,1]; neural network के softmax output से मिलाने के लिए ज़रूरी
seed = 7                                     # seed value 7 तय की — ताकि randomness reproducible (दोबारा उत्पन्न होने योग्य) रहे

X, Y = shuffle(X, y, random_state=seed)      # X और y की rows को एक साथ (जोड़ी बनाए रखते हुए) randomly फेरबदल किया, ताकि किसी क्रम/पैटर्न का असर training पर न पड़े


---------------------------------------------------------------------------
## **1. अलग-अलग Optimisation Algorithms को Tune करना**
-------------------------------------------------------------------

variable optimizer में नीचे दिए गए optimizers को एक list के रूप में डालें -
- SGD, RMSprop, Adam, Nadam

param_grid में parameter optimizer को dict का उपयोग करके optimizer के रूप में डालें


In [ ]:
optimizer = ['SGD', 'RMSprop', 'Adam', 'Nadam']     # आज़माने के लिए चार अलग-अलग optimizers की एक list बनाई — हर optimizer weights को अलग तरीके से update करता है

param_grid = dict(optimizer=optimizer)              # एक dictionary बनाई जिसमें key 'optimizer' है और value ऊपर वाली list — GridSearchCV इसी के अनुसार हर optimizer को आज़माएगा


### Model बनाएं

`create_model` function में Dense class का उपयोग करके एक fully-connected (पूरी तरह जुड़ा हुआ) network structure define करें
- एक sequential model बनाएं
- Model, 4 variables वाली data की rows को अपेक्षा (expect) करता है (input_dim=4 argument)
- पहली hidden layer में 64 nodes हैं और relu activation function उपयोग होता है
- दूसरी hidden layer में 32 nodes हैं और relu activation function उपयोग होता है
- तीसरी hidden layer में 16 nodes हैं और relu activation function उपयोग होता है
- Output layer में 3 nodes हैं और softmax activation function उपयोग होता है
- Model को compile करते समय निम्नलिखित parameters दें -

           -optimizer को optimizer के रूप में
           -loss को categorical cross entropy के रूप में
           -metrics को accuracy के रूप में
-  Compile किया हुआ model return करें


In [ ]:
def create_model(optimizer='adam'):                                          # model बनाने वाला function — बाहर से कौनसा optimizer उपयोग करना है यह parameter के रूप में लेता है
    model = Sequential()                                                     # एक खाली Sequential model बनाया — इसमें layers एक के बाद एक जोड़ी जाएँगी
    model.add(Dense(64, input_dim=4, activation='relu'))                     # पहली hidden layer जोड़ी — 64 neurons, input में 4 features आएंगे, relu activation
    model.add(Dense(32, activation='relu'))                                  # दूसरी hidden layer जोड़ी — 32 neurons, relu activation
    model.add(Dense(16, activation='relu'))                                  # तीसरी hidden layer जोड़ी — 16 neurons, relu activation
    model.add(Dense(3, activation='softmax'))                                # output layer जोड़ी — 3 neurons (3 species के लिए), softmax activation से probability distribution मिलता है
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])   # model को compile किया — दिया गया optimizer, multi-class के लिए categorical cross-entropy loss, और accuracy metric सेट किया
    return model                                                             # तैयार (compiled) model return किया


`KerasClassifier` function का उपयोग करके model function को निम्नलिखित parameters के साथ call करें -
- build_fn को create_model के रूप में
- batch_size को 10 के रूप में
- verbose को 0 के रूप में
- epochs को 10 के रूप में

ऊपर वाले को variable model में save करें


In [ ]:
model = KerasClassifier(build_fn=create_model, batch_size=10, verbose=0, epochs=10)   # create_model function को sklearn-compatible classifier में लपेटा (wrap) — batch_size=10 (हर बार 10 samples), verbose=0 (training के दौरान output print न हो), epochs=10 (data पर 10 बार training pass)


grid में `GridSearchCV` function का उपयोग करें और निम्नलिखित parameters दें -
- estimator को model के रूप में
- param_grid को param_grid के रूप में
- n_jobs को 1 के रूप में

अब X और Y के साथ grid का उपयोग करके model को fit करें और उसे grid_result में save करें

**नोट (Note)** - fit model चलाते समय इस cell को चलने में 2-5 मिनट तक लग सकते हैं


In [ ]:
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=1)      # GridSearchCV सेटअप किया — यह param_grid में दिए हर optimizer को बारी-बारी से आज़माकर सबसे अच्छा वाला ढूँढेगा; n_jobs=1 यानी एक समय में एक ही combination चलेगा
grid_result = grid.fit(X, Y)                                                # असली training/search चलाई — हर optimizer के साथ model को train करके performance compare किया, नतीजे grid_result में save हुए


### नतीजों (results) का सार (summary) देखने के लिए नीचे दिए गए cell को run करें


In [ ]:
# summarize results
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))     # सबसे अच्छा (best) score और उसे देने वाले parameters print किए
means = grid_result.cv_results_['mean_test_score']                                   # हर combination का औसत (mean) test score निकाला
stds = grid_result.cv_results_['std_test_score']                                     # हर combination के score का standard deviation (उतार-चढ़ाव) निकाला
params = grid_result.cv_results_['params']                                           # हर combination में उपयोग हुए parameters निकाले
for mean, stdev, param in zip(means, stds, params):                                  # तीनों lists को एक साथ (जोड़ी बनाकर) loop किया
    print("%f (%f) with: %r" % (mean, stdev, param))                                 # हर combination का mean, std deviation, और parameters print किए


### नतीजों को plot करने के लिए नीचे दिए गए cell को run करें


In [ ]:
pyplot.figure(figsize=(10,8))                                                # एक नया figure बनाया, आकार 10x8 inches
#pyplot.xticks(grid_result1.cv_results_['mean_test_score'])
pyplot.title("Performance metrics of each optimiser")                        # plot का शीर्षक (title) सेट किया
plot=sns.barplot(grid_result.cv_results_['mean_test_score'],optimizer)       # bar plot बनाया — हर optimizer का mean test score bar के रूप में दिखाया
pyplot.show()                                                                 # plot को screen पर दिखाया


---------------------------------------------------------------------------
## **2. SGD Optimizer की Learning Rate और Momentum को Tune करना**
-------------------------------------------------------------------
variable learn_rate में नीचे दी गई learning rates को एक list के रूप में डालें -
- 0.001, 0.01, 0.3

variable momentum में नीचे दिए गए momentum values को एक list के रूप में डालें -
- 0.0, 0.4, 0.9


In [ ]:
learn_rate = [0.001, 0.01, 0.3]     # आज़माने के लिए तीन अलग-अलग learning rates की list — यह तय करता है कि हर step में weights कितना बदलें
momentum = [0.0, 0.4, 0.9]          # आज़माने के लिए तीन अलग-अलग momentum values की list — पिछले updates की दिशा को कितना याद रखना है यह तय करता है


### Model बनाएं

ऊपर जैसे ही model parameters का उपयोग करके `create_model1` function में model बनाएं

variable optimizer में SGD optimizer का उपयोग करते हुए निम्नलिखित parameters दें -

     -lr को learn_rate के रूप में
     -momentum को momentum के रूप में
Model को compile करते समय निम्नलिखित parameters दें -

     -optimizer को optimizer के रूप में
     -loss को categorical cross entropy के रूप में
     -metrics को accuracy के रूप में
Compile किया हुआ model return करें


In [ ]:
def create_model1(learn_rate=0.01, momentum=0):                              # model बनाने वाला function — learning_rate और momentum बाहर से parameter के रूप में लेता है

    model = Sequential()                                                     # एक खाली Sequential model बनाया
    model.add(Dense(64, input_dim=4, activation='relu'))                     # पहली hidden layer — 64 neurons, 4 input features, relu activation
    model.add(Dense(32, activation='relu'))                                  # दूसरी hidden layer — 32 neurons, relu activation
    model.add(Dense(16, activation='relu'))                                  # तीसरी hidden layer — 16 neurons, relu activation
    model.add(Dense(3, activation='softmax'))                                # output layer — 3 neurons (3 species), softmax activation
    optimizer = SGD(lr=learn_rate, momentum=momentum)                        # SGD optimizer बनाया, दी गई learning_rate और momentum के साथ
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])   # model compile किया — यह SGD optimizer, categorical cross-entropy loss, aur accuracy metric के साथ

    return model                                                             # तैयार model return किया


`KerasClassifier` function का उपयोग करके model function को निम्नलिखित parameters के साथ call करें -

- build_fn को create_model1 के रूप में
- batch_size को 10 के रूप में
- verbose को 0 के रूप में
- epochs को 10 के रूप में

ऊपर वाले को variable model1 में save करें


In [ ]:
model1 = KerasClassifier(build_fn=create_model1, batch_size=10, verbose=0, epochs=10)   # create_model1 function को sklearn-compatible classifier में लपेटा — इस बार अलग-अलग learn_rate और momentum आज़माए जाएँगे


param_grid1 में parameter learn_rate को learn_rate के रूप में और momentum को momentum के रूप में dict का उपयोग करके डालें

grid1 में `GridSearchCV` function का उपयोग करें और निम्नलिखित parameters दें -
- estimator को model1 के रूप में
- param_grid को param_grid1 के रूप में
- n_jobs को 1 के रूप में


In [ ]:
param_grid1 = dict(learn_rate=learn_rate, momentum=momentum)                   # dictionary बनाई जिसमें learn_rate और momentum दोनों की lists हैं — GridSearchCV इनके सभी combinations आज़माएगा
grid1 = GridSearchCV(estimator=model1, param_grid=param_grid1, n_jobs=1)       # GridSearchCV सेटअप किया — model1 पर, param_grid1 के हर combination के साथ, n_jobs=1


seed value सेट करने के लिए numpy के `random.seed` function का उपयोग करें

अब X और Y के साथ grid1 का उपयोग करके model को fit करें और उसे grid_result1 में save करें

**नोट (Note)** - fit model चलाते समय इस cell को चलने में 2-5 मिनट तक लग सकते हैं


In [ ]:
np.random.seed(seed)                        # numpy का random seed फिर से तय किया, ताकि यह search भी reproducible (दोबारा उत्पन्न होने योग्य) रहे
grid_result1 = grid1.fit(X, Y)              # असली training/search चलाई — हर learn_rate-momentum combination को आज़माकर performance compare किया, नतीजे grid_result1 में save हुए


### नतीजों (results) का सार (summary) देखने के लिए नीचे दिए गए cell को run करें


In [ ]:
# summarize results
print("Best: %f using %s" % (grid_result1.best_score_, grid_result1.best_params_))    # सबसे अच्छा (best) score और उसे देने वाला learn_rate/momentum combination print किया
means1 = grid_result1.cv_results_['mean_test_score']                                  # हर combination का औसत (mean) test score निकाला
stds1 = grid_result1.cv_results_['std_test_score']                                    # हर combination के score का standard deviation निकाला
params1 = grid_result1.cv_results_['params']                                          # हर combination में उपयोग हुए parameters निकाले
for mean, stdev, param in zip(means1, stds1, params1):                                # तीनों lists को एक साथ loop किया
    print("%f (%f) with: %r" % (mean, stdev, param))                                  # हर combination का mean, std deviation, और parameters print किए


### नतीजों को plot करने के लिए नीचे दिए गए cell को run करें


In [ ]:
params1=pd.DataFrame(params1)                                                # params1 (dictionaries की list) को एक pandas DataFrame में बदला, ताकि columns के रूप में access कर सकें
pyplot.figure(figsize=(10,8))                                                # एक नया figure बनाया, आकार 10x8 inches
#pyplot.xticks(grid_result1.cv_results_['mean_test_score'])
pyplot.title("Performance metrics of SGD optimiser with different learning rates and momentum")   # plot का शीर्षक सेट किया
plot1=sns.barplot(params1["learn_rate"],grid_result1.cv_results_['mean_test_score'],hue=params1["momentum"])   # bar plot बनाया — x-axis पर learn_rate, y-axis पर score, अलग-अलग momentum को अलग रंग (hue) से दिखाया
plot1.set(ylabel='Score')                                                     # y-axis का label 'Score' रखा
pyplot.show()                                                                 # plot को screen पर दिखाया


---------------------------------------------------------------------------
## **3. Adam Optimizer के Beta Values को Tune करना**
-------------------------------------------------------------------

variable beta_1 में नीचे दी गई values को एक list के रूप में डालें -
- 0.001, 0.01, 0.3

variable beta_2 में नीचे दी गई values को एक list के रूप में डालें -
- 0.0, 0.4, 0.9


In [ ]:
beta_1 = [0.001, 0.01, 0.3]     # आज़माने के लिए beta_1 की तीन values की list — यह पिछले gradients के औसत (moving average) को कितना याद रखना है, यह नियंत्रित करता है
beta_2 = [0.0, 0.4, 0.9]        # आज़माने के लिए beta_2 की तीन values की list — यह पिछले gradients के वर्ग (squared) के औसत को कितना याद रखना है, यह नियंत्रित करता है


### Model बनाएं

ऊपर जैसे ही model parameters का उपयोग करके `create_model2` function में model बनाएं

variable optimizer में Adam optimizer का उपयोग करते हुए निम्नलिखित parameters दें -

     -beta_1 को beta_1 के रूप में
     -beta_2 को beta_2 के रूप में
Model को compile करते समय निम्नलिखित parameters दें -

     -optimizer को optimizer के रूप में
     -loss को categorical cross entropy के रूप में
     -metrics को accuracy के रूप में
Compile किया हुआ model return करें


In [ ]:
def create_model2(beta_1=0.01, beta_2=0):                                     # model बनाने वाला function — beta_1 और beta_2 बाहर से parameter के रूप में लेता है

    model = Sequential()                                                     # एक खाली Sequential model बनाया
    model.add(Dense(64, input_dim=4, activation='relu'))                     # पहली hidden layer — 64 neurons, 4 input features, relu activation
    model.add(Dense(32, activation='relu'))                                  # दूसरी hidden layer — 32 neurons, relu activation
    model.add(Dense(16, activation='relu'))                                  # तीसरी hidden layer — 16 neurons, relu activation
    model.add(Dense(3, activation='softmax'))                                # output layer — 3 neurons (3 species), softmax activation
    optimizer = Adam(beta_1=beta_1, beta_2=beta_2)                           # Adam optimizer बनाया, दिए गए beta_1 और beta_2 के साथ
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])   # model compile किया — Adam optimizer, categorical cross-entropy loss, और accuracy metric के साथ
    return model                                                             # तैयार model return किया


`KerasClassifier` function का उपयोग करके model function को निम्नलिखित parameters के साथ call करें -

- build_fn को create_model2 के रूप में
- batch_size को 10 के रूप में
- verbose को 0 के रूप में
- epochs को 10 के रूप में

ऊपर वाले को variable model2 में save करें


In [ ]:
model2 = KerasClassifier(build_fn=create_model2, batch_size=10, verbose=0, epochs=10)   # create_model2 function को sklearn-compatible classifier में लपेटा — इस बार अलग-अलग beta_1, beta_2 आज़माए जाएँगे


param_grid2 में parameter beta_1 को beta_1 के रूप में और beta_2 को beta_2 के रूप में dict का उपयोग करके डालें

grid2 में `GridSearchCV` function का उपयोग करें और निम्नलिखित parameters दें -
- estimator को model2 के रूप में
- param_grid को param_grid2 के रूप में
- n_jobs को 1 के रूप में


In [ ]:
param_grid2 = dict(beta_1=beta_1, beta_2=beta_2)                                # dictionary बनाई जिसमें beta_1 और beta_2 दोनों की lists हैं — GridSearchCV इनके सभी combinations आज़माएगा
grid2 = GridSearchCV(estimator=model2, param_grid=param_grid2, n_jobs=1)       # GridSearchCV सेटअप किया — model2 पर, param_grid2 के हर combination के साथ, n_jobs=1


seed value सेट करने के लिए numpy के `random.seed` function का उपयोग करें

अब X और Y के साथ grid2 का उपयोग करके model को fit करें और उसे grid_result2 में save करें

**नोट (Note)** - fit model चलाते समय इस cell को चलने में 2-5 मिनट तक लग सकते हैं


In [ ]:
np.random.seed(seed)                        # numpy का random seed फिर से तय किया, ताकि यह search भी reproducible रहे
grid_result2 = grid2.fit(X, Y)              # असली training/search चलाई — हर beta_1-beta_2 combination को आज़माकर performance compare किया, नतीजे grid_result2 में save हुए


### नतीजों (results) का सार (summary) देखने के लिए नीचे दिए गए cell को run करें


In [ ]:
# summarize results
print("Best: %f using %s" % (grid_result2.best_score_, grid_result2.best_params_))    # सबसे अच्छा (best) score और उसे देने वाला beta_1/beta_2 combination print किया
means2 = grid_result2.cv_results_['mean_test_score']                                  # हर combination का औसत (mean) test score निकाला
stds2 = grid_result2.cv_results_['std_test_score']                                    # हर combination के score का standard deviation निकाला
params2 = grid_result2.cv_results_['params']                                          # हर combination में उपयोग हुए parameters निकाले
for mean, stdev, param in zip(means2, stds2, params2):                                # तीनों lists को एक साथ loop किया
    print("%f (%f) with: %r" % (mean, stdev, param))                                  # हर combination का mean, std deviation, और parameters print किए


### नतीजों को plot करने के लिए नीचे दिए गए cell को run करें


In [ ]:
params2=pd.DataFrame(params2)                                                # params2 (dictionaries की list) को एक pandas DataFrame में बदला
pyplot.figure(figsize=(10,8))                                                # एक नया figure बनाया, आकार 10x8 inches
#pyplot.xticks(grid_result1.cv_results_['mean_test_score'])
pyplot.title("Performance metrics of SGD optimiser with different beta values")   # plot का शीर्षक सेट किया
plot2=sns.barplot(params2["beta_1"],grid_result2.cv_results_['mean_test_score'],hue=params2["beta_2"])   # bar plot बनाया — x-axis पर beta_1, y-axis पर score, अलग-अलग beta_2 को अलग रंग (hue) से दिखाया
plot2.set(ylabel='Score')                                                     # y-axis का label 'Score' रखा
pyplot.show()                                                                 # plot को screen पर दिखाया


**नोट (Note)**

इसी तरह आप बाकी optimization algorithms का भी उपयोग करके operations कर सकते हैं।

आप बाकी optimizers के parameters भी उसी तरह tune कर सकते हैं जैसे हमने इस exercise में किया।


### अपने scores और model को testing के लिए save करने के लिए नीचे दिए गए cells को run करें


In [ ]:
with open("score.txt","w") as f:                              # 'score.txt' फ़ाइल को write mode में खोला
    f.write(str(round(grid_result.best_score_,2)))          # पहले scenario (सभी optimizers) का best score, 2 decimal तक round करके फ़ाइल में लिखा
with open("params.txt","w") as f:                             # 'params.txt' फ़ाइल को write mode में खोला
    f.write(str(grid_result.best_params_))                   # पहले scenario के best parameters फ़ाइल में लिखे

with open("score1.txt","w") as f:                             # 'score1.txt' फ़ाइल को write mode में खोला
    f.write(str(round(grid_result1.best_score_,2)))          # दूसरे scenario (SGD learn_rate/momentum) का best score फ़ाइल में लिखा
with open("params1.txt","w") as f:                            # 'params1.txt' फ़ाइल को write mode में खोला
    f.write(str(grid_result1.best_params_))                   # दूसरे scenario के best parameters फ़ाइल में लिखे

with open("score2.txt","w") as f:                             # 'score2.txt' फ़ाइल को write mode में खोला
    f.write(str(round(grid_result2.best_score_,2)))          # तीसरे scenario (Adam beta values) का best score फ़ाइल में लिखा
with open("params2.txt","w") as f:                            # 'params2.txt' फ़ाइल को write mode में खोला
    f.write(str(grid_result2.best_params_))                   # तीसरे scenario के best parameters फ़ाइल में लिखे


In [ ]:
def save_model(model):                              # function जो trained model को disk पर save करता है
    # saving model
    json_model = model.to_json()                    # model के architecture (structure) को JSON format की string में बदला
    open('model.json', 'w').write(json_model)        # उस JSON structure को 'model.json' फ़ाइल में लिखा
    # saving weights
    model.save_weights('model.h5', overwrite=True)   # model के trained weights को 'model.h5' फ़ाइल में save किया (अगर पहले से मौजूद हो तो overwrite कर दिया)
classifier = grid_result.best_estimator_.model       # naya untrained model banane ki jagah, GridSearchCV (scenario 1) ka best (already-trained) model nikala — real seekhe hue (learned) weights ke saath
save_model(classifier)                               # ऊपर बनाए गए trained model का structure और weights save किए
